In [ ]:
import dxpy
from pyspark.sql import SparkSession
import hail as hl
import os

In [ ]:
builder = (
    SparkSession
    .builder
    .enableHiveSupport())
spark = builder.getOrCreate()
hl.init(sc=spark.sparkContext)

In [ ]:
files = [f'file:///mnt/project/Bulk/DRAGEN WGS/DRAGEN population level WGS variants, pVCF format - 500k release/chr4/ukb24310_c4_b{n}_v1.vcf.gz'
         for n in range(1018, 1119)] 

In [ ]:
mt = hl.import_vcf(
    files,
    reference_genome='GRCh38', force_bgz=True, array_elements_required=False)

In [ ]:
db_name = "hail_kcnip4"
mt_name = "kcnip4_raw.mt"

stmt = f"CREATE DATABASE IF NOT EXISTS {db_name} LOCATION 'dnax://'"
print(stmt)
spark.sql(stmt).show()

In [ ]:
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']

In [ ]:
url = f"dnax://{db_uri}/{mt_name}"

#mt.write(url)

In [ ]:
url # should be : dnax://database-J0kYjJ0JxzjV0VbPkP1K81p7/kcnip4_raw.mt

In [ ]:
mt = hl.read_matrix_table(url)

In [ ]:
mt.count()

### split multiallelic variants + do simple QC - call rate 0.99

In [ ]:
mt = hl.variant_qc(mt)

mt.variant_qc.call_rate.take(10)

mt = mt.filter_rows(mt.variant_qc.call_rate > 0.99)
mt = hl.split_multi_hts(mt)

db_name = "hail_kcnip4"
mt2_name = "kcnip4_split_filtered.mt"
url2 = f"dnax://{db_uri}/{mt2_name}"

#mt.write(url2)

mt = hl.read_matrix_table(url2)
mt.count()

mt = mt.drop(mt.variant_qc)
mt = hl.variant_qc(mt)
mt = mt.filter_rows(hl.min(mt.variant_qc.AC) > 20)

mt3_name = "kcnip4_ready.mt"
url3 = f"dnax://{db_uri}/{mt3_name}"

url3

#mt.write(url3)

mt = hl.read_matrix_table(url3)
mt.count()

mac_stats = hl.agg.stats(mt.variant_qc.AC[1])

In [ ]:
mt.aggregate_rows(mac_stats)

### calculate the LD with our focal SNP for the whole region

In [ ]:
my_snp = hl.locus('chr4', 21369602, reference_genome='GRCh38')

focal_mt = mt.filter_rows(mt.locus == my_snp)

if focal_mt.count_rows() != 1:
    raise ValueError('Could not find chr4:21 369 602')

focal_gt_tbl = (
    focal_mt
        .select_entries(fgt = focal_mt.GT.n_alt_alleles()) 
        .entries()
        .key_by('s')                                         
        .select('fgt')                                 
)

mt = mt.annotate_cols(fgt = focal_gt_tbl[mt.col_key].fgt)
mt = mt.annotate_rows(
    r2  = hl.agg.corr(mt.GT.n_alt_alleles(), mt.fgt)**2
)

ht = mt.rows()
ht = ht.key_by()
ht = ht.select(
    ht.locus,
    ht.alleles,
    ht.r2,
    ht.variant_qc.AF,
    ht.variant_qc.p_value_hwe
    
)

ld_name = "kcnip4_ld.mt"

url4 = f"dnax://{db_uri}/{ld_name}"

#ht.write(url4, overwrite = True)
ht = hl.read_table(url4)

In [ ]:
ht.filter(ht.r2 > 0.95).show()

In [ ]:
ht.filter(ht.r2 > 0.95).count()

In [ ]:
ht.show()

In [ ]:
ht.export('ld_to_chr4_21369602.tsv.bgz')

In [ ]:
%%bash

hdfs dfs -get ./*.bgz .

### first association study - just with "ever taken ARBs"- same subset as originally in the paper

In [ ]:
mt3_name = "kcnip4_ready.mt"
url3 = f"dnax://{db_uri}/{mt3_name}"
mt = hl.read_matrix_table(url3)
my_snp = hl.locus('chr4', 21369602, reference_genome='GRCh38')
focal_mt = mt.filter_rows(mt.locus == my_snp)

focal_gt_tbl = (
    focal_mt
        .select_entries(fgt = focal_mt.GT.n_alt_alleles()) 
        .entries()
        .key_by('s')                                         
        .select('fgt')                                 
)

mt = mt.annotate_cols(fgt = focal_gt_tbl[mt.col_key].fgt)
mt = mt.annotate_rows(
    r2  = hl.agg.corr(mt.GT.n_alt_alleles(), mt.fgt)**2
)

master_path = "file:///mnt/project/<PROJECT_FOLDER>/"
bp = hl.import_table(
    master_path+"bp_phenos_all.tsv",
    impute = True
)

bp = bp.transmute(s = hl.str(bp.eid))
bp = bp.key_by(bp.s)

mt_bp = mt.filter_cols(hl.is_defined(bp[mt.col_key]))

# implementing this filter : mt = mt.filter_rows(mt.r2 > 0.1) leaves me with 125 variants with lowest AF of 7% so I am skipping this

mt_bp = mt_bp.drop(mt_bp.variant_qc)
mt_bp = hl.variant_qc(mt_bp)

mt_bp = mt_bp.filter_rows(hl.min(mt_bp.variant_qc.AC) > 20)
mt_bp = mt_bp.annotate_cols(arb = bp[mt_bp.col_key].angiotensin_receptor_blocker)

mt_bp = mt_bp.annotate_entries(
    minor = hl.if_else(
        mt_bp.variant_qc.AF[1] <= 0.5, # this variant qc was calculated recently so the AF is correct          
        mt_bp.GT.n_alt_alleles(),                    
        2 - mt_bp.GT.n_alt_alleles()                 
    )
)

In [ ]:
mt_bp_name = "kcnip4_ever_on_arbs_paper_dataset.mt"
url5 = f"dnax://{db_uri}/{mt_bp_name}"
mt_bp.write(url5, overwrite = True)

mt_bp = hl.read_matrix_table(url5)
mt_bp.aggregate_rows(hl.agg.stats(mt_bp.variant_qc.AF[1]))
mt_bp.count()

In [ ]:
mt_bp.aggregate_cols(hl.agg.counter(mt_bp.arb))

In [ ]:
ever_on_arbs_unadj = hl.logistic_regression_rows(
    test='firth',
    y=mt_bp.arb,
    x=mt_bp.minors,
    covariates=[1],
    pass_through = ['mt_bp.r2']
)

logreg_1 = "kcnip4_on_arbs_unadjusted.mt"
url6 = f"dnax://{db_uri}/{logreg_1}"
ever_on_arbs_unadj.write(url6, overwrite = True)

In [ ]:
ever_on_arbs_unadj = hl.read_table(url6)

In [ ]:
my_snp = hl.locus('chr4', 21369602, reference_genome='GRCh38')
ever_on_arbs_unadj.filter(ever_on_arbs_unadj.locus == my_snp).show()

In [ ]:
ever_on_arbs_unadj.filter(ever_on_arbs_unadj.p_value < 0.00000001).count()

### second association study - with the ARB response phenotypes - same subset as originally in the paper

In [ ]:
mt3_name = "kcnip4_ready.mt"
url3 = f"dnax://{db_uri}/{mt3_name}"
mt = hl.read_matrix_table(url3)

master_path = "file:///mnt/project/<PROJECT_FOLDER>/"
arb = hl.import_table(
    master_path+"agg_arb_with_snps.csv",
    impute = True
)

to_drop = [f for f in arb.row if 'rs' in f]

to_drop.remove('arb_is_first_therapy')

arb = arb.drop(*to_drop)
arb = arb.transmute(s = hl.str(arb.eid))
arb = arb.key_by(arb.s)

mt_arb = mt.filter_cols(hl.is_defined(arb[mt.col_key]))
mt_arb = mt_arb.drop(mt_arb.variant_qc)
mt_arb = hl.variant_qc(mt_arb)
mt_arb = mt_arb.filter_rows(hl.min(mt_arb.variant_qc.AC) > 20)
mt_arb = mt_arb.annotate_cols(**arb[mt_arb.col_key])

In [ ]:
mt_arb_name = "kcnip4_arb_response_dataset.mt"
url7 = f"dnax://{db_uri}/{mt_arb_name}"

In [ ]:
mt_arb.write(url7)

In [ ]:
mt_arb = hl.read_matrix_table(url7)
mt_arb.count()

In [ ]:
# !!! make sure we are testing for the minor allele

mt_arb = mt_arb.annotate_entries(
    minor = hl.if_else(
        mt_arb.variant_qc.AF[1] <= 0.5, # this variant qc was calculated recently so the AF is correct          
        mt_arb.GT.n_alt_alleles(),                    
        2 - mt_arb.GT.n_alt_alleles()                 
    )
)

In [ ]:
linreg_phenos = ['num_arb_therapies',
 'num_diff_drug_classes',
 'num_diff_arbs',
 'longest_arb_duration',
 'avg_dose_Candesartan Cilexetil',
 'avg_dose_Eprosartan',
 'avg_dose_Irbesartan',
 'avg_dose_Losartan Potassium',
 'avg_dose_Olmesartan',
 'avg_dose_Sacubitril/Valsartan',
 'avg_dose_Telmisartan',
 'avg_dose_Valsartan',
 'num_arb_therapies_ADJ',
 'num_diff_drug_classes_ADJ',
 'num_diff_arbs_ADJ',
 'longest_arb_duration_ADJ',
 'avg_dose_Candesartan Cilexetil_ADJ',
 'avg_dose_Eprosartan_ADJ',
 'avg_dose_Irbesartan_ADJ',
 'avg_dose_Losartan Potassium_ADJ',
 'avg_dose_Olmesartan_ADJ',
 'avg_dose_Telmisartan_ADJ',
 'avg_dose_Valsartan_ADJ',
 'dose_ever_increased_ADJ',
 'arb_is_longest_therapy_ADJ',
 'arb_is_last_therapy_ADJ',
 'arb_ever_augmented_ADJ',
 'changed_from_arb_ADJ',
         ]

logreg_phenos = [
 'dose_ever_increased',
 'arb_is_longest_therapy',
 'arb_is_last_therapy',
 'arb_is_first_therapy',
 'arb_ever_augmented',
 'changed_from_arb'
         ]

In [ ]:
phenos_to_test = [mt_arb[ph] for ph in logreg_phenos]

In [ ]:
arb_response = hl.logistic_regression_rows(
    test='firth',
    y=phenos_to_test,
    x=mt_arb.minor,
    covariates=[1]
)

In [ ]:
logreg_2 = "kcnip4_arb_response_logreg.mt"
url8 = f"dnax://{db_uri}/{logreg_2}"
#arb_response.write(url8)

In [ ]:
phenos_to_test_lin = [mt_arb[ph] for ph in linreg_phenos]

In [ ]:
arb_response_lin = hl.linear_regression_rows(
    y=phenos_to_test_lin,
    x=mt_arb.minor,
    covariates=[1],
)

In [ ]:
arb_response_lin.show()

In [ ]:
linreg = "kcnip4_arb_response_linreg.mt"
url9 = f"dnax://{db_uri}/{linreg}"
#arb_response_lin.write(url9, overwrite = True)

In [ ]:
logreg = hl.read_table(url8)

In [ ]:
logreg.show()

In [ ]:
linreg = hl.read_table(url9)

In [ ]:
from hail.expr import types as t

In [ ]:
def explode_linear(ht, phen_list, *, model='linear'):
    array_fields = [
        name
        for name, typ in ht.row.dtype.items()
        if (isinstance(typ, t.tarray)
            and name not in ht.key
            and typ.element_type == t.tfloat64)          # adjust if int32/float32
    ]
    if not array_fields:
        raise RuntimeError("No numeric array-typed row fields found – "
                           "is this a linear_regression_rows table?")

    phen_lit = hl.literal({i: p for i, p in enumerate(phen_list)})

    ht = ht.annotate(
        per_pheno = hl.range(len(phen_list)).map(
            lambda i: hl.struct(
                phenotype = phen_lit[i],
                **{f: ht[f][i] for f in array_fields}
            )
        )
    )
    ht = ht.explode('per_pheno')
    ht = ht.transmute(**ht.per_pheno) 
    return ht.annotate(model = model)

def explode_logistic(ht, phen_list, *, model='logistic'):
    phen_lit = hl.literal({i: p for i, p in enumerate(phen_list)})

    ht = ht.annotate(
        per_pheno = hl.enumerate(ht.logistic_regression).map(
            lambda tpl: tpl[1].annotate(phenotype = phen_lit[tpl[0]])
        )
    )
    ht = ht.explode('per_pheno')
    ht = ht.transmute(**ht.per_pheno)
    return ht.annotate(model = model)

In [ ]:
# `phenos` must be the SAME order you passed into the regression call
log_res_long = explode_logistic(logreg, logreg_phenos, model='logistic')
lin_res_long = explode_linear (linreg, linreg_phenos, model='linear')

In [ ]:
combined = log_res_long.union(lin_res_long, unify = True)

In [ ]:
combined.show()

In [ ]:
my_snp = hl.locus('chr4', 21369602, reference_genome='GRCh38')
focal_mt = mt_arb.filter_rows(mt_arb.locus == my_snp)

focal_gt_tbl = (
    focal_mt
        .select_entries(fgt = focal_mt.GT.n_alt_alleles()) 
        .entries()
        .key_by('s')                                         
        .select('fgt')                                 
)

mt_arb = mt_arb.annotate_cols(fgt = focal_gt_tbl[mt_arb.col_key].fgt)
mt_arb = mt_arb.annotate_rows(
    r2  = hl.agg.corr(mt_arb.GT.n_alt_alleles(), mt_arb.fgt)**2
)

In [ ]:
combined = combined.transmute(
    r2 = mt_arb.rows()[combined.key].r2,
    AF = mt_arb.rows()[combined.key].variant_qc.AF
)

combined = combined.drop(combined.fit, combined.logistic_regression)

In [ ]:
combined.filter(combined.p_value < 0.00000001).show()

In [ ]:
combined_name = "arb_response_regression_results.mt"
url10 = f"dnax://{db_uri}/{combined_name}"
#combined.write(url10)

In [ ]:
combined = hl.read_table(url10)

In [ ]:
combined.filter(combined.p_value < 0.05).export("significant_arb_kcnip4_ukb.tsv")

In [ ]:
%%bash

hdfs dfs -get ./*.tsv .

In [ ]:
in_ld = combined.filter((combined.p_value < 0.05)&(combined.r2>0.1))

In [ ]:
in_ld = in_ld.filter(in_ld.phenotype.contains("_ADJ"))

In [ ]:
in_ld.export("arb_kcnip4_inld.tsv")

In [ ]:
%%bash

hdfs dfs -get ./arb_kcnip4_inld.tsv .